# LegalEagle — FastAPI + SQLite Backend (Notebook 6)

**4 Endpoints:**  `POST /upload`  ·  `POST /analyze`  ·  `GET /report/{id}`  ·  `POST /ask`

**Test without frontend:** After Cell 2, open `http://localhost:8000/docs` in your browser!

## 0 — Setup Paths + Verify Files

In [20]:
import os, sys, subprocess, time, json, httpx
from pathlib import Path

BACKEND_DIR = Path('..') / 'backend'
DATA_DIR    = Path('..') / 'data'

files = ['main.py', 'tasks.py', 'database.py', 'celery_app.py']
for f in files:
    p = BACKEND_DIR / f
    print(f'  {"OK" if p.exists() else "MISSING"} {p}')

(DATA_DIR / 'uploads').mkdir(parents=True, exist_ok=True)
print('Setup complete!')

  OK ..\backend\main.py
  OK ..\backend\tasks.py
  OK ..\backend\database.py
  OK ..\backend\celery_app.py
Setup complete!


## 1 — Start the FastAPI Server

Runs `uvicorn` in the background. The server auto-reloads on any code change.

**After this cell runs:** open `http://localhost:8000/docs` in your browser!

In [21]:
backend_abs = str(BACKEND_DIR.resolve())
venv_python = sys.executable

server_proc = subprocess.Popen(
    [venv_python, '-m', 'uvicorn', 'main:app',
     '--host', '0.0.0.0', '--port', '8000'],  # no --reload
    cwd=backend_abs,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

print(f'Server starting (PID {server_proc.pid})...')
time.sleep(5)

for attempt in range(12):
    try:
        r = httpx.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'Server READY! {r.json()}')
            print('Open: http://localhost:8000/docs')
            break
    except Exception:
        time.sleep(1)
else:
    print('Server did not start. Re-run this cell.')

Server starting (PID 19108)...
Server READY! {'status': 'ok', 'timestamp': '2026-06-11T10:06:01.002199'}
Open: http://localhost:8000/docs


## 2 — `POST /upload` — Upload a Contract

Saves the file to `data/uploads/` and returns a unique `job_id`.

We'll use the **RISKY consulting agreement** — designed to trigger HIGH risk scores.

In [22]:
BASE_URL = 'http://localhost:8000'

# Use the risky demo contract
risky = DATA_DIR / 'sample_contracts' / 'RISKY_ConsultingAgreement_TechVentures.txt'
if not risky.exists():
    # Fall back to first sample contract
    risky = sorted((DATA_DIR / 'sample_contracts').glob('*.txt'))[0]

print(f'Uploading: {risky.name}')
with open(risky, 'rb') as f:
    resp = httpx.post(
        f'{BASE_URL}/upload',
        files={'file': (risky.name, f, 'text/plain')},
        timeout=30
    )

print(f'HTTP {resp.status_code}')
upload_data = resp.json()
print(json.dumps(upload_data, indent=2))
JOB_ID = upload_data['job_id']
print(f'\nJob ID: {JOB_ID}')

Uploading: RISKY_ConsultingAgreement_TechVentures.txt
HTTP 200
{
  "job_id": "751cbe66-b90a-4265-ac59-8146e505c214",
  "filename": "RISKY_ConsultingAgreement_TechVentures.txt",
  "path": "data\\uploads\\751cbe66-b90a-4265-ac59-8146e505c214.txt",
  "message": "File uploaded. Call POST /analyze with this job_id to start analysis."
}

Job ID: 751cbe66-b90a-4265-ac59-8146e505c214


## 3 — `POST /analyze` — Run AI Analysis

Runs the full pipeline synchronously:
**BERT NER → Qdrant RAG comparison → LLM risk scoring → SQLite save**

⏳ This takes **2-5 minutes** on first run (models loading). Subsequent runs are faster.

In [23]:
print('Starting analysis... (this runs BERT + LLM + Qdrant — be patient!)')
print('ETA: 2-5 minutes on first run')
print()

start = time.time()
resp = httpx.post(
    f'{BASE_URL}/analyze',
    json={'job_id': JOB_ID},
    timeout=600  # 10 minutes max
)
elapsed = time.time() - start

print(f'HTTP {resp.status_code}  (took {elapsed:.0f}s)')
if resp.status_code == 200 and resp.content:
    data = resp.json()
    print(json.dumps(data, indent=2))
elif not resp.content:
    print('Empty response — server may have restarted.')
    print('Stop old server (Cell 8), re-run Cell 1, then re-run Cell 2 & 3.')
else:
    print(f'Error: {resp.status_code}')
    print(resp.text[:500])

Starting analysis... (this runs BERT + LLM + Qdrant — be patient!)
ETA: 2-5 minutes on first run

HTTP 200  (took 37s)
{
  "job_id": "751cbe66-b90a-4265-ac59-8146e505c214",
  "status": "done",
  "overall_score": 6.0,
  "message": "Analysis complete! Call GET /report/{job_id} for full results."
}


## 4 — `GET /report/{id}` — Fetch Results from SQLite

In [24]:
from IPython.display import Markdown, display

resp = httpx.get(f'{BASE_URL}/report/{JOB_ID}', timeout=10)
data = resp.json()

print(f'Contract  : {data["contract_name"]}')
print(f'Status    : {data["status"]}')
print(f'Score     : {data.get("overall_score", "N/A")}/10')
print(f'HR Review : {data.get("needs_human_review", False)}')

print('\nExtracted Entities:')
for k, v in data.get('entities', {}).items():
    print(f'  {k:20s}: {v[:2]}')

print('\nRisk Scores (sorted by score):')
for clause, d in sorted(data.get('risk_scores', {}).items(), key=lambda x: -x[1]['score']):
    bar = '##' * d['score'] + '..' * (10 - d['score'])
    flag = ' *** HIGH RISK' if d['score'] > 7 else ''
    print(f'  [{bar}] {d["score"]}/10  {clause}{flag}')

if data.get('report_markdown'):
    print('\n--- Full Rendered Report ---')
    display(Markdown(data['report_markdown']))

Contract  : 751cbe66-b90a-4265-ac59-8146e505c214
Status    : done
Score     : 6.0/10
HR Review : True

Extracted Entities:
  Termination         : ['termination clause present']
  Indemnification     : ['indemnification clause present']
  Non_Compete         : ['non-compete clause present']
  Governing_Law       : ['governing law clause present']
  Confidentiality     : ['confidentiality clause present']

Risk Scores (sorted by score):
  [################....] 8/10  Non_Compete *** HIGH RISK
  [##############......] 7/10  Indemnification
  [############........] 6/10  Termination
  [##########..........] 5/10  Confidentiality
  [########............] 4/10  Governing_Law

--- Full Rendered Report ---


# Legal Contract Risk Analysis Report
**Contract:** 751cbe66-b90a-4265-ac59-8146e505c214
**Generated:** 2026-06-11 10:06 UTC
**Overall Risk Score:** 6.0/10  🟡 MEDIUM

---
## Summary
Analyzed **5 clause types**.
**1 HIGH RISK** clause(s) detected.

> ⚠️ **ATTORNEY REVIEW REQUIRED**
> High-risk clauses flagged:
> - **Non_Compete**: 8/10

---
## Clause Analysis
### Non_Compete ⚠️
**Score:** `[████████░░]` 8/10  🔴 HIGH
**Text:** _non-compete clause present_
**Reasoning:** <sentence>

### Indemnification
**Score:** `[███████░░░]` 7/10  🟡 MEDIUM
**Text:** _indemnification clause present_
**Reasoning:** <sentence>

### Termination
**Score:** `[██████░░░░]` 6/10  🟡 MEDIUM
**Text:** _termination clause present_
**Reasoning:** <sentence>

### Confidentiality
**Score:** `[█████░░░░░]` 5/10  🟡 MEDIUM
**Text:** _confidentiality clause present_
**Reasoning:** <sentence>

### Governing_Law
**Score:** `[████░░░░░░]` 4/10  🟡 MEDIUM
**Text:** _governing law clause present_
**Reasoning:** <sentence>

---
## Recommendations
- 🔴 **Non_Compete** (8/10): Seek legal advice before signing.
- 🔴 **Indemnification** (7/10): Seek legal advice before signing.
- 🟡 **Termination** (6/10): Review carefully.
- 🟡 **Confidentiality** (5/10): Review carefully.
- 🟡 **Governing_Law** (4/10): Review carefully.

## 5 — `POST /ask` — Q&A over the Contract

In [25]:
questions = [
    'Who are the parties in this contract?',
    'What is the termination clause?',
    'What governing law applies?',
    'Are there any non-compete restrictions?',
    'What is the overall risk score?',
]

for q in questions:
    r = httpx.post(
        f'{BASE_URL}/ask',
        json={'job_id': JOB_ID, 'question': q},
        timeout=10
    )
    if r.status_code == 200:
        print(f'Q: {q}')
        print(f'A: {r.json()["answer"]}')
    else:
        print(f'Q: {q}')
        print(f'ERROR {r.status_code}: {r.json()}')
    print()

Q: Who are the parties in this contract?
A: Not found in this contract's analysis.

Q: What is the termination clause?
A: Extracted entities: termination clause present
Termination: score 6/10 — <sentence> | Extracted: termination clause present
From report: **Text:** _termination clause present_ | - 🟡 **Termination** (6/10): Review carefully.

Q: What governing law applies?
A: Extracted entities: governing law clause present
Governing_Law: score 4/10 — <sentence> | Extracted: governing law clause present
From report: **Text:** _governing law clause present_ | - 🟡 **Governing_Law** (4/10): Review carefully.

Q: Are there any non-compete restrictions?
A: Extracted entities: non-compete clause present
Non_Compete: score 8/10 — <sentence> | Extracted: non-compete clause present
From report: **Text:** _non-compete clause present_

Q: What is the overall risk score?
A: From report: **Overall Risk Score:** 6.0/10  🟡 MEDIUM | **1 HIGH RISK** clause(s) detected.



## 6 — Inspect SQLite Database

All results are in `data/legaleagle.db`.

In [26]:
import sqlite3
db_path = str((DATA_DIR / 'legaleagle.db').resolve())
con = sqlite3.connect(db_path)
con.row_factory = sqlite3.Row

print('=== analysis_jobs ===')
for r in con.execute('SELECT id, contract_name, status, overall_score, needs_human_review FROM analysis_jobs').fetchall():
    print(f'  [{r["status"]:10s}] score={r["overall_score"]}  hr={bool(r["needs_human_review"])}  {r["contract_name"][:40]}')

print('\n=== qa_records (last 8) ===')
for r in con.execute('SELECT question, answer FROM qa_records ORDER BY id DESC LIMIT 8').fetchall():
    print(f'  Q: {r["question"][:60]}')
    print(f'  A: {r["answer"][:100]}')
    print()
con.close()
print(f'DB: {db_path}')

=== analysis_jobs ===

=== qa_records (last 8) ===
DB: C:\Users\Saahil Saitwal\OneDrive\Desktop\LegalEagle\data\legaleagle.db


## 7 — Interactive Swagger UI Links

In [27]:
from IPython.display import HTML
display(HTML(
    '<h3>Test all 4 endpoints interactively in your browser:</h3>'
    '<p><a href="http://localhost:8000/docs" target="_blank" style="font-size:18px;color:green">'
    '🚀 Open Swagger UI → http://localhost:8000/docs</a></p>'
    '<p><a href="http://localhost:8000/redoc" target="_blank" style="font-size:16px">'
    '📄 Open ReDoc → http://localhost:8000/redoc</a></p>'
))

## 8 — Stop Server

In [28]:
server_proc.terminate()
print('Server stopped. Results saved in data/legaleagle.db')

Server stopped. Results saved in data/legaleagle.db


---
✅ FastAPI + SQLite backend fully working end-to-end.